# Ficheros · XML, ZIP y utilidades

Ejercicios cortos de manejo de ficheros con la biblioteca estándar: XML con `xml.etree.ElementTree`, un mismo registro cargado desde JSON y desde XML, compresión con `zipfile`, copias binarias comprobadas con `hashlib`, utilidades de texto (frecuencias, filtrado de logs, partir y unir ficheros, renombrar en bloque), una calculadora que evalúa expresiones sin `eval`, errores de JSON y un sorteo a partir de un CSV. Todos los ficheros se crean desde el propio notebook en la carpeta `datos_utilidades`. Ideas de [PYnative](https://pynative.com/python-exercises-with-solutions/) y de la hoja de ruta de [retosdeprogramacion.com](https://retosdeprogramacion.com/roadmap/).

In [1]:
import ast
import csv
import hashlib
import json
import operator
import random
import re
import shutil
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from dataclasses import dataclass
from itertools import batched
from pathlib import Path

CARPETA = Path("datos_utilidades")
# Se parte siempre de una carpeta vacía para que el notebook dé lo mismo en cada ejecución.
shutil.rmtree(CARPETA, ignore_errors=True)
CARPETA.mkdir()

## 1. Crear un fichero XML

Crear con `ElementTree` el fichero `persona.xml` con los datos de una programadora (nombre, edad y ciudad, y un atributo `id`) y la lista de lenguajes que conoce, cada uno con su nivel y el año en que empezó como atributos. Sangrar el árbol con `ET.indent` y mostrar el fichero.

In [2]:
datos = {
    "id": "7",
    "nombre": "Nayra Santana",
    "edad": 29,
    "ciudad": "Las Palmas de Gran Canaria",
    "lenguajes": [
        ("Python", "avanzado", 2016),
        ("SQL", "avanzado", 2017),
        ("JavaScript", "intermedio", 2019),
        ("TypeScript", "básico", 2023),
        ("Rust", "básico", 2024),
    ],
}

persona = ET.Element("persona", id=datos["id"])
for campo in ("nombre", "edad", "ciudad"):
    ET.SubElement(persona, campo).text = str(datos[campo])
lenguajes = ET.SubElement(persona, "lenguajes")
for nombre, nivel, desde in datos["lenguajes"]:
    ET.SubElement(lenguajes, "lenguaje", nivel=nivel, desde=str(desde)).text = nombre

arbol = ET.ElementTree(persona)
ET.indent(arbol)
arbol.write(CARPETA / "persona.xml", encoding="utf-8", xml_declaration=True)

texto_xml = (CARPETA / "persona.xml").read_text(encoding="utf-8")
print(texto_xml)
assert texto_xml.count("<lenguaje ") == len(datos["lenguajes"])

<?xml version='1.0' encoding='utf-8'?>
<persona id="7">
  <nombre>Nayra Santana</nombre>
  <edad>29</edad>
  <ciudad>Las Palmas de Gran Canaria</ciudad>
  <lenguajes>
    <lenguaje nivel="avanzado" desde="2016">Python</lenguaje>
    <lenguaje nivel="avanzado" desde="2017">SQL</lenguaje>
    <lenguaje nivel="intermedio" desde="2019">JavaScript</lenguaje>
    <lenguaje nivel="básico" desde="2023">TypeScript</lenguaje>
    <lenguaje nivel="básico" desde="2024">Rust</lenguaje>
  </lenguajes>
</persona>


## 2. Leer y recorrer el XML

Leer `persona.xml` con `ET.parse` y recorrer el árbol: mostrar la etiqueta y el texto de los hijos directos de la raíz y, después, una tabla con cada lenguaje, su nivel y los años de experiencia en 2026. Comprobar que los datos leídos coinciden con los escritos.

In [3]:
raiz = ET.parse(CARPETA / "persona.xml").getroot()
print(f"<{raiz.tag}> con atributos {raiz.attrib}")
for hijo in raiz:
    texto = hijo.text.strip() if hijo.text and hijo.text.strip() else f"({len(hijo)} hijos)"
    print(f"  {hijo.tag:<10}{texto}")

print(f"\n{'Lenguaje':<12}{'Nivel':<12}{'Años':>5}")
for lenguaje in raiz.iter("lenguaje"):
    print(f"{lenguaje.text:<12}{lenguaje.get('nivel'):<12}{2026 - int(lenguaje.get('desde')):>5}")

leidos = [(l.text, l.get("nivel"), int(l.get("desde"))) for l in raiz.iter("lenguaje")]
assert leidos == datos["lenguajes"]
assert int(raiz.findtext("edad")) == datos["edad"]

<persona> con atributos {'id': '7'}
  nombre    Nayra Santana
  edad      29
  ciudad    Las Palmas de Gran Canaria
  lenguajes (5 hijos)

Lenguaje    Nivel        Años
Python      avanzado       10
SQL         avanzado        9
JavaScript  intermedio      7
TypeScript  básico          3
Rust        básico          2


## 3. Buscar y modificar

Con el mismo árbol, buscar con `find`, `findall` y expresiones XPath sencillas: el nombre, todos los lenguajes, los de nivel avanzado, el segundo lenguaje de la lista y los empezados después de 2018. Subir el nivel de TypeScript a intermedio, añadir Go como lenguaje básico y guardar el resultado en `persona_modificada.xml`. Volver a leerlo para comprobar los cambios.

In [4]:
print("Nombre:          ", raiz.find("nombre").text)
print("Todos:           ", [l.text for l in raiz.findall("lenguajes/lenguaje")])
print("Avanzados:       ", [l.text for l in raiz.findall(".//lenguaje[@nivel='avanzado']")])
print("Segundo:         ", raiz.find("lenguajes/lenguaje[2]").text)
# ElementTree no admite comparaciones numéricas en XPath: se filtra en Python.
recientes = [l.text for l in raiz.iter("lenguaje") if int(l.get("desde")) > 2018]
print("Desde 2019:      ", recientes)
assert recientes == ["JavaScript", "TypeScript", "Rust"]

raiz.find(".//lenguaje[.='TypeScript']").set("nivel", "intermedio")
ET.SubElement(raiz.find("lenguajes"), "lenguaje", nivel="básico", desde="2026").text = "Go"
arbol_mod = ET.ElementTree(raiz)
ET.indent(arbol_mod)
arbol_mod.write(CARPETA / "persona_modificada.xml", encoding="utf-8", xml_declaration=True)

releida = ET.parse(CARPETA / "persona_modificada.xml").getroot()
niveles = {l.text: l.get("nivel") for l in releida.iter("lenguaje")}
print("\nNiveles tras el cambio:", niveles)
assert niveles["TypeScript"] == "intermedio" and niveles["Go"] == "básico"
assert len(niveles) == len(datos["lenguajes"]) + 1

Nombre:           Nayra Santana
Todos:            ['Python', 'SQL', 'JavaScript', 'TypeScript', 'Rust']
Avanzados:        ['Python', 'SQL']
Segundo:          SQL
Desde 2019:       ['JavaScript', 'TypeScript', 'Rust']

Niveles tras el cambio: {'Python': 'avanzado', 'SQL': 'avanzado', 'JavaScript': 'intermedio', 'TypeScript': 'intermedio', 'Rust': 'básico', 'Go': 'básico'}


## 4. Un registro desde JSON y desde XML

Guardar los mismos datos en `persona.json`. Crear una clase `Programadora` con dos constructores alternativos, `desde_json` y `desde_xml` (métodos de clase), que lean cada formato y devuelvan un objeto. Comprobar que los dos objetos son iguales.

In [5]:
(CARPETA / "persona.json").write_text(json.dumps(datos, ensure_ascii=False, indent=2), encoding="utf-8")


@dataclass(frozen=True)
class Programadora:
    id: int
    nombre: str
    edad: int
    ciudad: str
    lenguajes: tuple[tuple[str, str, int], ...]

    @classmethod
    def desde_json(cls, ruta):
        d = json.loads(Path(ruta).read_text(encoding="utf-8"))
        return cls(int(d["id"]), d["nombre"], d["edad"], d["ciudad"],
                   tuple((n, nivel, desde) for n, nivel, desde in d["lenguajes"]))

    @classmethod
    def desde_xml(cls, ruta):
        r = ET.parse(ruta).getroot()
        return cls(int(r.get("id")), r.findtext("nombre"), int(r.findtext("edad")), r.findtext("ciudad"),
                   tuple((l.text, l.get("nivel"), int(l.get("desde"))) for l in r.iter("lenguaje")))


desde_json = Programadora.desde_json(CARPETA / "persona.json")
desde_xml = Programadora.desde_xml(CARPETA / "persona.xml")
print(desde_json)
print("¿Iguales?", desde_json == desde_xml)
assert desde_json == desde_xml
assert desde_xml != Programadora.desde_xml(CARPETA / "persona_modificada.xml")

Programadora(id=7, nombre='Nayra Santana', edad=29, ciudad='Las Palmas de Gran Canaria', lenguajes=(('Python', 'avanzado', 2016), ('SQL', 'avanzado', 2017), ('JavaScript', 'intermedio', 2019), ('TypeScript', 'básico', 2023), ('Rust', 'básico', 2024)))
¿Iguales? True


## 5. Comprimir con `zipfile`

Crear tres ficheros muy distintos (un texto muy repetitivo, un CSV y bytes aleatorios) y comprimirlos en `paquete.zip` con `ZIP_DEFLATED`. Listar el contenido con el tamaño original, el comprimido y el porcentaje de ahorro de cada fichero, y el ratio total. Extraer solo el CSV a otra carpeta y comprobar que es idéntico al original.

In [6]:
generador = random.Random(11)
originales = {
    "repetitivo.txt": ("Sol y playa en Fuerteventura.\n" * 2_000).encode("utf-8"),
    "temperaturas.csv": "".join(f"dia_{d},{generador.uniform(17, 29):.1f}\n" for d in range(1, 366)).encode("utf-8"),
    "aleatorio.bin": generador.randbytes(20_000),
}
for nombre, contenido in originales.items():
    (CARPETA / nombre).write_bytes(contenido)

ruta_zip = CARPETA / "paquete.zip"
with zipfile.ZipFile(ruta_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for nombre in originales:
        z.write(CARPETA / nombre, arcname=nombre)

with zipfile.ZipFile(ruta_zip) as z:
    print(f"{'Fichero':<18}{'Original':>10}{'Comprimido':>12}{"Ahorro":>9}")
    for info in z.infolist():
        ahorro = 1 - info.compress_size / info.file_size
        print(f"{info.filename:<18}{info.file_size:>10}{info.compress_size:>12}{ahorro:>9.2%}")
    total = sum(i.file_size for i in z.infolist())
    comprimido = sum(i.compress_size for i in z.infolist())
    print(f"Ratio total: {total / comprimido:.2f} : 1")
    assert z.testzip() is None
    z.extract("temperaturas.csv", CARPETA / "extraido")

extraido = (CARPETA / "extraido" / "temperaturas.csv").read_bytes()
print("¿El CSV extraído es idéntico?", extraido == originales["temperaturas.csv"])
assert extraido == originales["temperaturas.csv"]
# Los bytes aleatorios apenas se comprimen; el texto repetitivo, muchísimo.
with zipfile.ZipFile(ruta_zip) as z:
    assert z.getinfo("repetitivo.txt").compress_size < 0.01 * len(originales["repetitivo.txt"])
    assert z.getinfo("aleatorio.bin").compress_size > 0.99 * len(originales["aleatorio.bin"])

Fichero             Original  Comprimido   Ahorro
repetitivo.txt         60000         197   99.67%
temperaturas.csv        4637        1359   70.69%
aleatorio.bin          20000       20010   -0.05%
Ratio total: 3.92 : 1
¿El CSV extraído es idéntico? True


## 6. Copiar por bloques y comprobar con `hashlib`

Copiar el fichero binario del ejercicio anterior leyéndolo en bloques de 4096 bytes, sin cargarlo entero en memoria, y contar los bloques. Comprobar con el resumen SHA-256 que la copia es idéntica. Escribir también funciones que den el tamaño de un fichero en KB y digan si un fichero está vacío.

In [7]:
TAM_BLOQUE = 4096


def copiar_por_bloques(origen, destino, tam=TAM_BLOQUE):
    bloques = 0
    with open(origen, "rb") as fo, open(destino, "wb") as fd:
        while bloque := fo.read(tam):
            fd.write(bloque)
            bloques += 1
    return bloques


def sha256(ruta):
    with open(ruta, "rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def tamano_kb(ruta):
    return Path(ruta).stat().st_size / 1024


def esta_vacio(ruta):
    return Path(ruta).stat().st_size == 0


origen, copia = CARPETA / "aleatorio.bin", CARPETA / "aleatorio_copia.bin"
n = copiar_por_bloques(origen, copia)
print(f"Bloques copiados: {n} (el último incompleto: {origen.stat().st_size % TAM_BLOQUE} bytes)")
print("SHA-256 original:", sha256(origen)[:16], "…")
print("SHA-256 copia:   ", sha256(copia)[:16], "…")
print(f"Tamaño: {tamano_kb(copia):.2f} KB")

(CARPETA / "vacio.txt").touch()
for nombre in ("vacio.txt", "aleatorio_copia.bin"):
    print(f"¿{nombre} está vacío? {esta_vacio(CARPETA / nombre)}")

assert sha256(origen) == sha256(copia) and n == -(-20_000 // TAM_BLOQUE)
assert esta_vacio(CARPETA / "vacio.txt") and not esta_vacio(copia)
assert tamano_kb(copia) == 20_000 / 1024

Bloques copiados: 5 (el último incompleto: 3616 bytes)
SHA-256 original: 41c62e92d1b0af6a …
SHA-256 copia:    41c62e92d1b0af6a …
Tamaño: 19.53 KB
¿vacio.txt está vacío? True
¿aleatorio_copia.bin está vacío? False


## 7. Frecuencia de letras

Guardar un texto en `texto.txt` y contar con `Counter` cuántas veces aparece cada letra, sin distinguir mayúsculas de minúsculas e ignorando espacios, signos y cifras. Mostrar las 6 letras más frecuentes con su porcentaje y una barra.

In [8]:
texto = """La Graciosa es la isla habitada más pequeña de Canarias.
No tiene carreteras asfaltadas: sus calles son de arena.
Desde Caleta de Sebo se ve el risco de Famara, en Lanzarote.
En 2018 fue reconocida como la octava isla del archipiélago."""
(CARPETA / "texto.txt").write_text(texto, encoding="utf-8")

letras = Counter(c for c in (CARPETA / "texto.txt").read_text(encoding="utf-8").lower() if c.isalpha())
total_letras = letras.total()
print(f"{total_letras} letras, {len(letras)} distintas")
for letra, veces in letras.most_common(6):
    print(f"{letra}  {veces:>3}  {veces / total_letras:>6.1%}  {'█' * veces}")

assert total_letras == sum(c.isalpha() for c in texto)
assert letras["a"] == texto.lower().count("a")

185 letras, 24 distintas
a   35   18.9%  ███████████████████████████████████
e   26   14.1%  ██████████████████████████
s   17    9.2%  █████████████████
l   13    7.0%  █████████████
o   12    6.5%  ████████████
r   11    5.9%  ███████████


## 8. Filtrar los errores de un log

Crear `servidor.log` con líneas de nivel `INFO`, `WARNING` y `ERROR`, y copiar solo las líneas `ERROR` a `errores.log`, leyendo el original línea a línea. Mostrar cuántas líneas hay de cada nivel.

In [9]:
lineas_log = [
    "2026-11-02 08:00:01 INFO arranque del servidor de reservas",
    "2026-11-02 08:03:12 INFO reserva 1021 confirmada (Tejeda)",
    "2026-11-02 08:05:40 WARNING respuesta lenta de la pasarela de pago",
    "2026-11-02 08:06:02 ERROR pago rechazado en la reserva 1022",
    "2026-11-02 08:10:15 INFO reserva 1023 confirmada (Agaete)",
    "2026-11-02 08:12:33 ERROR tiempo de espera agotado con la base de datos",
    "2026-11-02 08:12:34 INFO reintento correcto",
    "2026-11-02 08:20:00 ERROR fichero de tarifas no encontrado",
]
(CARPETA / "servidor.log").write_text("\n".join(lineas_log) + "\n", encoding="utf-8")

niveles_log = Counter()
with (open(CARPETA / "servidor.log", encoding="utf-8") as entrada,
      open(CARPETA / "errores.log", "w", encoding="utf-8") as salida):
    for linea in entrada:
        nivel = linea.split()[2]
        niveles_log[nivel] += 1
        if nivel == "ERROR":
            salida.write(linea)

print(dict(niveles_log))
print((CARPETA / "errores.log").read_text(encoding="utf-8"))
assert (CARPETA / "errores.log").read_text(encoding="utf-8").count("\n") == niveles_log["ERROR"] == 3

{'INFO': 4, 'WARNING': 1, 'ERROR': 3}
2026-11-02 08:06:02 ERROR pago rechazado en la reserva 1022
2026-11-02 08:12:33 ERROR tiempo de espera agotado con la base de datos
2026-11-02 08:20:00 ERROR fichero de tarifas no encontrado



## 9. Partir un fichero y volver a unirlo

Crear `largo.txt` con 47 líneas y dividirlo en ficheros de 10 líneas (`parte_01.txt`, `parte_02.txt`…). Después unir las partes en orden en `unido.txt` y comprobar que es idéntico al original.

In [10]:
(CARPETA / "largo.txt").write_text("".join(f"línea {i:02d} del fichero\n" for i in range(1, 48)), encoding="utf-8")


def partir(ruta, lineas_por_parte=10):
    partes = []
    with open(ruta, encoding="utf-8") as f:
        for i, grupo in enumerate(batched(f, lineas_por_parte), start=1):
            parte = ruta.with_name(f"parte_{i:02d}.txt")
            parte.write_text("".join(grupo), encoding="utf-8")
            partes.append(parte)
    return partes


def unir(partes, destino):
    with open(destino, "w", encoding="utf-8") as salida:
        for parte in partes:
            salida.write(parte.read_text(encoding="utf-8"))


partes = partir(CARPETA / "largo.txt")
for parte in partes:
    print(f"{parte.name}: {len(parte.read_text(encoding='utf-8').splitlines())} líneas")
unir(sorted(CARPETA.glob("parte_*.txt")), CARPETA / "unido.txt")
iguales = (CARPETA / "unido.txt").read_bytes() == (CARPETA / "largo.txt").read_bytes()
print("¿Unido idéntico al original?", iguales)
assert len(partes) == 5 and iguales

parte_01.txt: 10 líneas
parte_02.txt: 10 líneas
parte_03.txt: 10 líneas
parte_04.txt: 10 líneas
parte_05.txt: 7 líneas
¿Unido idéntico al original? True


## 10. Reemplazar, fusionar e invertir

Con dos ficheros de texto pequeños: reemplazar en el primero una palabra por otra (solo la palabra completa, no dentro de otras), fusionar los dos en uno y escribir otro fichero con las líneas del fusionado en orden inverso.

In [11]:
(CARPETA / "ruta_a.txt").write_text("Vistas al mar desde Arucas\nParada en Firgas\nPaseo con la marea baja y baño en el mar\n", encoding="utf-8")
(CARPETA / "ruta_b.txt").write_text("Subida a Valleseco\nLlegada a Teror\n", encoding="utf-8")


def reemplazar_palabra(ruta, vieja, nueva):
    texto, cambios = re.subn(rf"\b{re.escape(vieja)}\b", nueva, ruta.read_text(encoding="utf-8"))
    ruta.write_text(texto, encoding="utf-8")
    return cambios


print("Cambios:", reemplazar_palabra(CARPETA / "ruta_a.txt", "mar", "océano"))

with open(CARPETA / "fusionado.txt", "w", encoding="utf-8") as salida:
    for nombre in ("ruta_a.txt", "ruta_b.txt"):
        salida.write((CARPETA / nombre).read_text(encoding="utf-8"))

lineas_fusion = (CARPETA / "fusionado.txt").read_text(encoding="utf-8").splitlines()
(CARPETA / "invertido.txt").write_text("\n".join(reversed(lineas_fusion)) + "\n", encoding="utf-8")

print("Fusionado:\n" + (CARPETA / "fusionado.txt").read_text(encoding="utf-8"))
print("Invertido:\n" + (CARPETA / "invertido.txt").read_text(encoding="utf-8"))
assert (CARPETA / "ruta_a.txt").read_text(encoding="utf-8").count("océano") == 2
assert "marea" in lineas_fusion[2]
assert (CARPETA / "invertido.txt").read_text(encoding="utf-8").splitlines()[::-1] == lineas_fusion

Cambios: 2
Fusionado:
Vistas al océano desde Arucas
Parada en Firgas
Paseo con la marea baja y baño en el océano
Subida a Valleseco
Llegada a Teror

Invertido:
Llegada a Teror
Subida a Valleseco
Paseo con la marea baja y baño en el océano
Parada en Firgas
Vistas al océano desde Arucas



## 11. Añadir un prefijo a los `.txt` de una carpeta

En una carpeta de pruebas con varios ficheros, añadir el prefijo `2026_` a todos los `.txt` que no lo tengan ya, sin tocar los demás. Guardar en `listado.txt` el contenido final de la carpeta, ordenado, y comprobar que repetir la operación no cambia nada.

In [12]:
notas = CARPETA / "notas"
notas.mkdir()
for nombre in ("compra.txt", "ideas.txt", "2026_viaje.txt", "foto.jpg", "resumen.md"):
    (notas / nombre).write_text(f"contenido de {nombre}\n", encoding="utf-8")


def anadir_prefijo(carpeta, prefijo="2026_"):
    renombrados = []
    for ruta in sorted(carpeta.glob("*.txt")):
        if not ruta.name.startswith(prefijo):
            ruta.rename(ruta.with_name(prefijo + ruta.name))
            renombrados.append(ruta.name)
    return renombrados


print("Renombrados:", anadir_prefijo(notas))
listado = sorted(p.name for p in notas.iterdir())
(CARPETA / "listado.txt").write_text("\n".join(listado) + "\n", encoding="utf-8")
print((CARPETA / "listado.txt").read_text(encoding="utf-8"))

assert anadir_prefijo(notas) == []
assert all(n.startswith("2026_") for n in listado if n.endswith(".txt"))
assert "foto.jpg" in listado and len(listado) == 5

Renombrados: ['compra.txt', 'ideas.txt']
2026_compra.txt
2026_ideas.txt
2026_viaje.txt
foto.jpg
resumen.md



## 12. Calculadora sin `eval`

`operaciones.txt` contiene una operación aritmética por línea (`3 + 4 * 2`). Evaluarlas sin `eval`: analizar cada línea con `ast.parse(..., mode="eval")` y recorrer el árbol permitiendo solo números, paréntesis y los operadores `+ - * / // % **`. Mostrar el resultado de cada línea válida e informar de las mal formadas (sintaxis incorrecta, nombres, llamadas o división entre cero) con su número de línea.

In [13]:
(CARPETA / "operaciones.txt").write_text("""3 + 4 * 2
(3 + 4) * 2
10 / 4
17 // 5 + 17 % 5
-2 ** 4
2 ** 10 - 24
7 +
precio * 2
__import__('os').getcwd()
1 / (3 - 3)
1.5 * (2 + -0.5)
""", encoding="utf-8")

OPERADORES = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def evaluar_nodo(nodo):
    match nodo:
        case ast.Constant(value=int() | float() as valor) if not isinstance(valor, bool):
            return valor
        case ast.BinOp(left=izq, op=op, right=der) if type(op) in OPERADORES:
            return OPERADORES[type(op)](evaluar_nodo(izq), evaluar_nodo(der))
        case ast.UnaryOp(op=op, operand=operando) if type(op) in OPERADORES:
            return OPERADORES[type(op)](evaluar_nodo(operando))
        case _:
            raise ValueError(f"elemento no permitido: {type(nodo).__name__}")


def calcular(expresion):
    return evaluar_nodo(ast.parse(expresion, mode="eval").body)


resultados, malas = {}, {}
with open(CARPETA / "operaciones.txt", encoding="utf-8") as f:
    for numero, linea in enumerate(f, start=1):
        linea = linea.strip()
        try:
            resultados[numero] = calcular(linea)
            print(f"{numero:>2}  {linea:<28} = {resultados[numero]}")
        except (SyntaxError, ValueError, ZeroDivisionError) as e:
            malas[numero] = type(e).__name__
            print(f"{numero:>2}  {linea:<28} ✗ {type(e).__name__}: {e}")

assert list(resultados.values()) == [11, 14, 2.5, 5, -16, 1000, 2.25]
assert malas == {7: "SyntaxError", 8: "ValueError", 9: "ValueError", 10: "ZeroDivisionError"}

 1  3 + 4 * 2                    = 11
 2  (3 + 4) * 2                  = 14
 3  10 / 4                       = 2.5
 4  17 // 5 + 17 % 5             = 5
 5  -2 ** 4                      = -16
 6  2 ** 10 - 24                 = 1000
 7  7 +                          ✗ SyntaxError: invalid syntax (<unknown>, line 1)
 8  precio * 2                   ✗ ValueError: elemento no permitido: Name
 9  __import__('os').getcwd()    ✗ ValueError: elemento no permitido: Call
10  1 / (3 - 3)                  ✗ ZeroDivisionError: division by zero
11  1.5 * (2 + -0.5)             = 2.25


## 13. JSON mal formado

Intentar leer `pedido.json`, al que le falta una coma, y capturar `json.JSONDecodeError` para informar del mensaje, la línea y la columna del error, mostrando esa línea con una marca debajo de la columna. Corregir el fichero y comprobar que ya se lee bien.

In [14]:
mal_formado = """{
  "cliente": "Guachinche El Lagar",
  "fecha": "2026-11-20"
  "lineas": [
    {"producto": "vino del país", "unidades": 12}
  ]
}"""
(CARPETA / "pedido.json").write_text(mal_formado, encoding="utf-8")


def leer_json(ruta):
    texto = Path(ruta).read_text(encoding="utf-8")
    try:
        return json.loads(texto)
    except json.JSONDecodeError as e:
        print(f"{e.msg} en la línea {e.lineno}, columna {e.colno}:")
        print("   " + texto.splitlines()[e.lineno - 1])
        print("   " + " " * (e.colno - 1) + "^")
        return None


assert leer_json(CARPETA / "pedido.json") is None
(CARPETA / "pedido.json").write_text(mal_formado.replace('"2026-11-20"\n', '"2026-11-20",\n'), encoding="utf-8")
pedido = leer_json(CARPETA / "pedido.json")
print("\nCorregido:", pedido)
assert pedido["lineas"][0]["unidades"] == 12

try:
    json.loads(mal_formado)
except json.JSONDecodeError as e:
    assert (e.lineno, e.colno) == (4, 3)

Expecting ',' delimiter en la línea 4, columna 3:
     "lineas": [
     ^

Corregido: {'cliente': 'Guachinche El Lagar', 'fecha': '2026-11-20', 'lineas': [{'producto': 'vino del país', 'unidades': 12}]}


## 14. Todos los valores de una clave en un JSON anidado

Escribir un generador que recorra cualquier estructura de diccionarios y listas cargada desde JSON y devuelva todos los valores de una clave, esté al nivel que esté. Usarlo para obtener todos los precios de la carta de un restaurante y sumarlos.

In [15]:
carta = {
    "restaurante": "Casa Mana",
    "precio": None,
    "secciones": [
        {"nombre": "Entrantes", "platos": [
            {"nombre": "papas arrugadas", "precio": 5.5},
            {"nombre": "queso asado", "precio": 7.0, "extras": [{"nombre": "mojo verde", "precio": 1.0}]},
        ]},
        {"nombre": "Principales", "platos": [
            {"nombre": "sancocho", "precio": 14.0},
            {"nombre": "conejo en salmorejo", "precio": 13.5},
        ]},
        {"nombre": "Postres", "platos": [{"nombre": "bienmesabe", "precio": 4.5}]},
    ],
}
(CARPETA / "carta.json").write_text(json.dumps(carta, ensure_ascii=False, indent=2), encoding="utf-8")


def valores_de(estructura, clave):
    match estructura:
        case dict():
            for k, v in estructura.items():
                if k == clave:
                    yield v
                yield from valores_de(v, clave)
        case list():
            for elemento in estructura:
                yield from valores_de(elemento, clave)


carta_leida = json.loads((CARPETA / "carta.json").read_text(encoding="utf-8"))
precios = [p for p in valores_de(carta_leida, "precio") if p is not None]
print("Precios:", precios)
print("Suma:", sum(precios))
print("Nombres:", list(valores_de(carta_leida, "nombre")))
assert precios == [5.5, 7.0, 1.0, 14.0, 13.5, 4.5]
assert len(list(valores_de(carta_leida, "nombre"))) == 9

Precios: [5.5, 7.0, 1.0, 14.0, 13.5, 4.5]
Suma: 45.5
Nombres: ['Entrantes', 'papas arrugadas', 'queso asado', 'mojo verde', 'Principales', 'sancocho', 'conejo en salmorejo', 'Postres', 'bienmesabe']


## 15. Sorteo desde un CSV

`participantes.csv` tiene el nombre, el municipio y el estado (`activo`, `inactivo` o `baja`) de cada participante de un sorteo. Leerlo con `csv.DictReader` y elegir con una semilla fija 3 ganadores distintos solo entre los activos. Si hay menos activos que premios, avisar con un error.

In [16]:
filas = [
    ("Airam", "Gáldar", "activo"), ("Yaiza", "Arrecife", "inactivo"), ("Dácil", "La Laguna", "activo"),
    ("Echedey", "Adeje", "baja"), ("Tanausú", "Los Llanos", "activo"), ("Guacimara", "Telde", "activo"),
    ("Iballa", "San Sebastián", "activo"), ("Jonay", "Valverde", "inactivo"), ("Chaxiraxi", "Candelaria", "activo"),
]
with open(CARPETA / "participantes.csv", "w", encoding="utf-8", newline="") as f:
    escritor = csv.writer(f)
    escritor.writerow(["nombre", "municipio", "estado"])
    escritor.writerows(filas)


def sortear(ruta, premios, semilla):
    with open(ruta, encoding="utf-8", newline="") as f:
        activos = [fila for fila in csv.DictReader(f) if fila["estado"] == "activo"]
    if len(activos) < premios:
        raise ValueError(f"hay {len(activos)} participantes activos para {premios} premios")
    return random.Random(semilla).sample(activos, premios)


ganadores = sortear(CARPETA / "participantes.csv", 3, semilla=2026)
for puesto, g in enumerate(ganadores, start=1):
    print(f"{puesto}.º {g['nombre']} ({g['municipio']})")

try:
    sortear(CARPETA / "participantes.csv", 7, semilla=2026)
except ValueError as e:
    print("No se puede sortear:", e)

assert len({g["nombre"] for g in ganadores}) == 3
assert all(g["estado"] == "activo" for g in ganadores)
assert ganadores == sortear(CARPETA / "participantes.csv", 3, semilla=2026)

1.º Airam (Gáldar)
2.º Tanausú (Los Llanos)
3.º Chaxiraxi (Candelaria)
No se puede sortear: hay 6 participantes activos para 7 premios
